In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]
df_day1 = spark.createDataFrame(
passengers_day1,
columns
)

In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
passengers_day2,
columns
)

In [0]:
df_day1.write \
    .format("delta") \
    .saveAsTable("passengers_delta")

In [0]:

df_day1.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
passenger_df = spark.read.table("passengers_delta")

display(passenger_df)

passenger_id,passenger_name,city,travel_class,country
101,Rahul Sharma,Hyderabad,Economy,India
102,Priya Reddy,Bangalore,Business,India
103,Amit Kumar,Mumbai,Economy,India
104,Sneha Patel,Delhi,Premium Economy,India
105,Farhan Ali,Chennai,Economy,India


In [0]:

%sql
DESCRIBE HISTORY passengers_delta
     

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-17T11:36:54.000Z,144705204055008,azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(238493463939051),5fb203f4-6f08-4d14-8a6b-027a045bee30,0617-113050-vqtue95h-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1848)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "passengers_delta"
)

delta_table.alias("target") \
.merge(
    df_day2.alias("source"),
    "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.read.table("passengers_delta") \
.filter("passenger_id = 106") \
.show()

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:
%sql
SELECT *
FROM passengers_delta VERSION AS OF 0
WHERE passenger_id = 104

passenger_id,passenger_name,city,travel_class,country
104,Sneha Patel,Delhi,Premium Economy,India


In [0]:
spark.read.table("passengers_delta") \
.filter("passenger_id = 104") \
.show()

+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         104|   Sneha Patel|Hyderabad|Premium Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
%sql
OPTIMIZE passengers_delta;
     

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781697117138, 1781697117828, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE passengers_delta
ZORDER BY (city);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1900), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781697178858, 1781697179478, 8, 0, null, List(0, 0), null, 5, 5, 0, 0, null, null)"


In [0]:
%sql
DELETE FROM passengers_delta
WHERE passenger_id = 105;

num_affected_rows
1


In [0]:
%sql
DESCRIBE HISTORY passengers_delta
     

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-17T11:53:48.000Z,144705204055008,azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(238493463939051),5bb7da68-8667-4bde-99f5-5d59879cf900,0617-113050-vqtue95h-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1900, p25FileSize -> 1880, numDeletionVectorsRemoved -> 1, minFileSize -> 1880, numAddedFiles -> 1, maxFileSize -> 1880, p75FileSize -> 1880, p50FileSize -> 1880, numAddedBytes -> 1880)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-17T11:53:47.000Z,144705204055008,azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(passenger_id#12435L = 105)""])",null,List(238493463939051),5bb7da68-8667-4bde-99f5-5d59879cf900,0617-113050-vqtue95h-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1646, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1232, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 412)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-17T11:41:46.000Z,144705204055008,azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(238493463939051),bcce22e3-3229-4b09-b414-fe8951d65b9c,0617-113050-vqtue95h-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 5, numRemovedBytes -> 8391, p25FileSize -> 1900, numDeletionVectorsRemoved -> 1, minFileSize -> 1900, numAddedFiles -> 1, maxFileSize -> 1900, p75FileSize -> 1900, p50FileSize -> 1900, numAddedBytes -> 1900)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-17T11:41:44.000Z,144705204055008,azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(passenger_id#11552L = passenger_id#11577L)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(238493463939051),bcce22e3-3229-4b09-b414-fe8951d65b9c,0617-113050-vqtue95h-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 4, numTargetBytesAdded -> 6543, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 2, executionTimeMs -> 5331, materializeSourceTimeMs -> 395, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2379, numTargetRowsUpdated -> 2, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 4, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2456)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-17T11:36:54.000Z,144705204055008,azuser7210_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(238493463939051),5fb203f4-6f08-4d14-8a6b-027a045bee30,0617-113050-vqtue95h-v2n,null,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 5, numOutputBytes -> 1848)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:

%sql
VACUUM passengers_delta;

path
""
